In [1]:
import pandas as pd
from pathlib import Path

# Project root
project_root = Path.cwd().parent

# Load datasets
train_df = pd.read_parquet(project_root / "data" / "splits" / "train.parquet")
val_df = pd.read_parquet(project_root / "data" / "splits" / "val.parquet")
test_df = pd.read_parquet(project_root / "data" / "splits" / "test.parquet")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

train_df.head()

Train: (5530, 6)
Validation: (1185, 6)
Test: (1185, 6)


,item_id,title,product_type,brand,color,target_category
0,B078GQNS66,"206 Collective Cameron Flat Thong Sandal, Nude...",SANDAL,206 Collective,Nude Patent Leather,Footwear
1,B07FBKDSNH,Rivet Hillmoor Mid-Century Queen Bed,HOME_FURNITURE_AND_DECOR,Rivet,Light Gray,Furniture
2,B07KCQX8QJ,Stone & Beam Lottie - Alfombra de lana tradici...,HOME_FURNITURE_AND_DECOR,Stone & Beam,None,Furniture
3,B07KXZ22GQ,Stone & Beam Westcott Modern Westcott Sofa Fabric,SOFA,Stone & Beam,None,Furniture
4,B07WSPFQBD,Klepe Men's Grey Chunky/Platform Sneakers-8 UK...,SHOES,Klepe,ग्रे(स्लेटी),Footwear


In [2]:
# Features and labels
X_train = train_df["title"]
y_train = train_df["target_category"]

X_val = val_df["title"]
y_val = val_df["target_category"]

X_test = test_df["title"]
y_test = test_df["target_category"]

print("Sample title:")
print(X_train.iloc[0])
print()
print("Label:", y_train.iloc[0])

Sample title:
206 Collective Cameron Flat Thong Sandal, Nude Patent Leather, 7.5 M US

Label: Footwear


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Build pipeline
baseline_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=20000,
        ngram_range=(1, 2)
    )),
    ("clf", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Train
baseline_model.fit(X_train, y_train)

print("Baseline model trained successfully!")

Baseline model trained successfully!


In [4]:
from sklearn.metrics import accuracy_score, classification_report

# Validation predictions
val_pred = baseline_model.predict(X_val)

# Accuracy
val_acc = accuracy_score(y_val, val_pred)

print(f"Validation Accuracy: {val_acc:.4f}")
print()
print("Classification Report:")
print(classification_report(y_val, val_pred))

Validation Accuracy: 0.9595

Classification Report:
              precision    recall  f1-score   support

        Bags       0.98      0.87      0.93       135
    Footwear       0.99      1.00      0.99       450
   Furniture       0.95      0.97      0.96       450
 Kitchenware       0.88      0.89      0.89       150

    accuracy                           0.96      1185
   macro avg       0.95      0.93      0.94      1185
weighted avg       0.96      0.96      0.96      1185



In [5]:
# Test predictions
test_pred = baseline_model.predict(X_test)

# Test accuracy
test_acc = accuracy_score(y_test, test_pred)

print(f"Test Accuracy: {test_acc:.4f}")

Test Accuracy: 0.9654


In [6]:
import joblib

model_path = project_root / "models" / "baseline" / "tfidf_logreg.pkl"

# Create folder if it doesn't exist
model_path.parent.mkdir(parents=True, exist_ok=True)

# Save model
joblib.dump(baseline_model, model_path)

print(f"Baseline model saved to: {model_path}")

Baseline model saved to: d:\christ college\sem 4\project\multimodel project\models\baseline\tfidf_logreg.pkl


In [7]:
baseline_metrics = {
    "model": "TF-IDF + Logistic Regression",
    "validation_accuracy": round(val_acc, 4),
    "test_accuracy": round(test_acc, 4)
}

print(baseline_metrics)

{'model': 'TF-IDF + Logistic Regression', 'validation_accuracy': 0.9595, 'test_accuracy': 0.9654}
